# Arena epsilon sweep — the edge-orbit residual

**A-Mark9 / NEXUS · arena programme · 2026-09-15**
Dean Kulik, QuHarmonics Research Group · ORCID 0009-0003-3128-8828

Run from live output. Every number below was produced by the code in this notebook.

---

## 0. Where this sits

Last run measured the invariant Hodge decomposition on three arenas and found the fork
was the wrong dichotomy: $\mathcal H^1_{\rm inv}=0$ everywhere, but $Q_3$ still carried a
non-exact invariant 1-form, sitting in $\operatorname{im}\delta$ rather than $\mathcal H^1$.
The open question was **what distinguishes $Q_3$**. Bipartiteness was the cheapest guess.

This run answers it. The discriminator is pure orbit combinatorics of the arena 1-skeleton,
visible before any Hodge theory:

$$\boxed{\ \epsilon \;=\; e_G^{\rm ns} \;-\; v_G \;+\; 1\ }$$

where $v_G$ counts $\operatorname{Aut}$-orbits of arena vertices and $e_G^{\rm ns}$ counts
orbits of arena edges **whose stabiliser contains no orientation-reversing element**.

---

## 1. What is theorem and what is measurement

This distinction matters and was blurred in the crosstalk. Three of the identities below are
**forced**, not discovered. The sweep confirms the implementation; it does not confirm them.

**T1.** $\dim (C^1)^G = e_G^{\rm ns}$. Orbit–stabiliser on oriented edges: an invariant
antisymmetric edge function has one free scalar per edge orbit, except that a reversing
stabiliser element forces $\omega(e)=\omega(-e)=-\omega(e)$, hence $\omega(e)=0$.

**T2.** $\dim (\operatorname{im}d_0)^G = v_G - 1$ for connected arenas. Invariant vertex
functions are constant on orbits; $d_0$ kills the global constant.

**T3.** $\epsilon = \dim\mathcal H^1_G + \dim(\operatorname{im}\delta)^G$. The Hodge splitting
$C^1 = \operatorname{im}d_0 \oplus \mathcal H^1 \oplus \operatorname{im}\delta$ is orthogonal and
$G$-equivariant, so it restricts to the invariant subspace. Subtracting T2 from T1 gives T3
immediately.

A corollary that also needs stating, because it looked like evidence last run and isn't:
the **circulation rank of the invariant sector equals $\epsilon$ automatically**. The kernel of
the pairing $C^1 \times Z_1 \to \mathbb R$ is exactly $\operatorname{im}d_0$, so
$\operatorname{rank} = \dim(C^1)^G - \dim(\operatorname{im}d_0)^G = \epsilon$. Observing nonzero
circulation whenever $\epsilon>0$ is not confirmation of anything.

**What is genuinely measured:** the values of $\epsilon$ across arenas, and — the real content —
**how $\epsilon$ splits between $\mathcal H^1_G$ and $(\operatorname{im}\delta)^G$**, which is the
only place parameter 7 can enter.

---

## 2. Method

Arena: unlabelled $K$-token configuration space over base graph $X$, exclusion, single-token
step moves — frozen parameters 1–6. Both open 2-cell predicates are run on every arena:
`disjoint` (Abrams–Ghrist, disjoint-support commuting moves) and `four_cycle` (fill every
4-cycle of the arena graph). Squares are deduplicated by corner set. All ranks use an
**absolute** singular-value tolerance of $10^{-9}$ — the locked correction; a relative tolerance
reports full rank on a matrix of $10^{-16}$ entries.


## 3. Library


In [ ]:
import itertools, numpy as np
from collections import defaultdict

ATOL = 1e-9

def rank_abs(M):
    if M.size == 0 or min(M.shape) == 0: return 0
    return int((np.linalg.svd(M, compute_uv=False) > ATOL).sum())

def colspace(M):
    if M.size == 0 or min(M.shape) == 0: return np.zeros((M.shape[0], 0))
    U, s, _ = np.linalg.svd(M, full_matrices=False)
    return U[:, s > ATOL]

def nullspace(M, n):
    if M.size == 0 or min(M.shape) == 0: return np.eye(n)
    _, s, Vt = np.linalg.svd(M, full_matrices=True)
    sf = np.zeros(Vt.shape[0]); sf[:len(s)] = s
    return Vt[sf <= ATOL].T

def arena(adj, N, K):
    verts = sorted((frozenset(s) for s in itertools.combinations(range(N), K)), key=lambda s: sorted(s))
    vi = {v: i for i, v in enumerate(verts)}
    und, nbr = set(), defaultdict(set)
    for x in verts:
        for u in x:
            for w in adj[u]:
                if w not in x:
                    y = frozenset((x - {u}) | {w})
                    und.add(frozenset((x, y))); nbr[x].add(y); nbr[y].add(x)
    edges = sorted((tuple(sorted(e, key=lambda s: sorted(s))) for e in und),
                   key=lambda p: (sorted(p[0]), sorted(p[1])))
    ei = {}
    for i, (a, b) in enumerate(edges):
        ei[(a, b)] = (i, 1); ei[(b, a)] = (i, -1)
    return verts, vi, edges, ei, nbr

def cells_disjoint(verts, nbr, adj):
    out, seen = [], set()
    for x in verts:
        mv = [(u, w) for u in x for w in adj[u] if w not in x]
        for (u1, w1), (u2, w2) in itertools.combinations(mv, 2):
            if {u1, w1} & {u2, w2}: continue
            x1 = frozenset((x - {u1}) | {w1}); x2 = frozenset((x - {u2}) | {w2})
            x12 = frozenset((x - {u1, u2}) | {w1, w2})
            key = tuple(sorted(tuple(sorted(v)) for v in (x, x1, x12, x2)))
            if key in seen: continue
            seen.add(key); out.append((x, x1, x12, x2))
    return out

def cells_fourcycle(verts, nbr):
    out, seen = [], set()
    for x, z in itertools.combinations(verts, 2):
        common = sorted(nbr[x] & nbr[z], key=lambda s: sorted(s))
        for a, b in itertools.combinations(common, 2):
            key = tuple(sorted(tuple(sorted(v)) for v in (x, a, z, b)))
            if key in seen: continue
            seen.add(key); out.append((x, a, z, b))
    return out

def d0_matrix(verts, vi, edges, ei):
    D = np.zeros((len(edges), len(verts)))
    for (a, b) in edges:
        i, _ = ei[(a, b)]; D[i, vi[b]] += 1; D[i, vi[a]] -= 1
    return D

def d1_matrix(cells, edges, ei):
    D = np.zeros((len(cells), len(edges)))
    for r, c in enumerate(cells):
        for k in range(4):
            i, s = ei[(c[k], c[(k + 1) % 4])]; D[r, i] += s
    return D

def edge_action(g, edges, ei):
    P = np.zeros((len(edges), len(edges)))
    for (a, b) in edges:
        i, _ = ei[(a, b)]
        j, s = ei[(frozenset(g[e] for e in a), frozenset(g[e] for e in b))]
        P[j, i] = s
    return P

def orbit_counts(verts, edges, AUT):
    seen, v_orb = set(), 0
    for x in verts:
        if x in seen: continue
        seen |= {frozenset(g[e] for e in x) for g in AUT}; v_orb += 1
    seen, e_orb, e_sr = set(), 0, 0
    for (a, b) in edges:
        key = frozenset((a, b))
        if key in seen: continue
        orb = {frozenset((frozenset(g[e] for e in a), frozenset(g[e] for e in b))) for g in AUT}
        seen |= orb; e_orb += 1
        if any(frozenset(g[e] for e in a) == b and frozenset(g[e] for e in b) == a for g in AUT):
            e_sr += 1
    return v_orb, e_orb, e_sr

def row(name, adj, N, K, AUT, pred):
    verts, vi, edges, ei, nbr = arena(adj, N, K)
    cells = cells_disjoint(verts, nbr, adj) if pred == "disjoint" else cells_fourcycle(verts, nbr)
    D0, D1 = d0_matrix(verts, vi, edges, ei), d1_matrix(cells, edges, ei)
    assert rank_abs(D1 @ D0) == 0
    Pi = sum(edge_action(g, edges, ei) for g in AUT) / len(AUT)
    inv = lambda B: rank_abs(Pi @ B) if B.shape[1] else 0
    imd0, imdel = colspace(D0), colspace(D1.T)
    H = nullspace(np.vstack([D1, D0.T]), len(edges))
    v_orb, e_orb, e_sr = orbit_counts(verts, edges, AUT)
    e_ns = e_orb - e_sr
    eps = e_ns - v_orb + 1
    return dict(arena=name, K=K, pred=pred, V=len(verts), E=len(edges), cells=len(cells),
                v_orb=v_orb, e_orb=e_orb, e_sr=e_sr, e_ns=e_ns, eps=eps,
                C1inv=rank_abs(Pi), exinv=inv(imd0), Hinv=inv(H), delinv=inv(imdel),
                H=H.shape[1])

def cyc(N):  return {v: {(v - 1) % N, (v + 1) % N} for v in range(N)}
def aut_cyc(N):
    return [tuple((i + k) % N for i in range(N)) for k in range(N)] + \
           [tuple((k - i) % N for i in range(N)) for k in range(N)]
Q3 = {v: {v ^ (1 << i) for i in range(3)} for v in range(8)}
def aut_q3():
    return [tuple(sum(((v >> i) & 1) << p[i] for i in range(3)) ^ m for v in range(8))
            for p in itertools.permutations(range(3)) for m in range(8)]
K9 = {v: set(range(9)) - {v} for v in range(9)}
def agl23():
    pts = [(a, b) for a in range(3) for b in range(3)]
    idx = {p: i for i, p in enumerate(pts)}
    out = []
    for (a, b, c, d) in itertools.product(range(3), repeat=4):
        if (a * d - b * c) % 3 == 0:
            continue
        for t in pts:
            out.append(tuple(idx[((a * p[0] + b * p[1] + t[0]) % 3,
                                  (c * p[0] + d * p[1] + t[1]) % 3)] for p in pts))
    return out

PET_V = sorted(itertools.combinations(range(5), 2))
PET_I = {s: i for i, s in enumerate(PET_V)}
PET = {PET_I[s]: {PET_I[t] for t in PET_V if not set(s) & set(t)} for s in PET_V}
def aut_pet():
    return [tuple(PET_I[tuple(sorted((p[s[0]], p[s[1]])))] for s in PET_V)
            for p in itertools.permutations(range(5))]
P8 = {v: {w for w in (v - 1, v + 1) if 0 <= w <= 7} for v in range(8)}
AUT_P8 = [tuple(range(8)), tuple(range(7, -1, -1))]

FAMILY = [("C6", cyc(6), 6, aut_cyc(6), [2, 3, 4]),
          ("C8", cyc(8), 8, aut_cyc(8), [2, 3, 4, 5]),
          ("Q3", Q3, 8, aut_q3(), [2, 3, 4]),
          ("Petersen", PET, 10, aut_pet(), [2, 3]),
          ("K9=Z3^2", K9, 9, agl23(), [2, 3]),
          ("P8 (|Aut|=2)", P8, 8, AUT_P8, [3])]

## 4. Sweep


In [ ]:
rows = []
for name, adj, N, AUT, Ks in FAMILY:
    for K in Ks:
        for pred in ("disjoint", "four_cycle"):
            r = row(name, adj, N, K, AUT, pred)
            rows.append(r)
            print(f"{r['arena']:<13} K={K} {pred:<11} eps={r['eps']:<3} "
                  f"Hinv={r['Hinv']} delinv={r['delinv']} "
                  f"T1={'ok' if r['C1inv']==r['e_ns'] else 'X'} "
                  f"T2={'ok' if r['exinv']==r['v_orb']-1 else 'X'} "
                  f"T3={'ok' if r['eps']==r['Hinv']+r['delinv'] else 'X'}")


## 5. Results — 30 rows, live

| base $X$ | $K$ | $V$ | $E$ | cells (dsj / 4cyc) | $v_G$ | $e_G$ | $e_G^{\rm sr}$ | $e_G^{\rm ns}$ | $\epsilon$ | $\mathcal H^1_G$ | $(\operatorname{im}\delta)^G$ |
|---|--:|--:|--:|--:|--:|--:|--:|--:|--:|--:|--:|
| $C_6$ | 2 | 15 | 24 | 9 / 9 | 3 | 2 | 0 | 2 | **0** | 0 | 0 |
| $C_6$ | 3 | 20 | 36 | 18 / 18 | 3 | 4 | 2 | 2 | **0** | 0 | 0 |
| $C_6$ | 4 | 15 | 24 | 9 / 9 | 3 | 2 | 0 | 2 | **0** | 0 | 0 |
| $C_8$ | 2 | 28 | 48 | 20 / 20 | 4 | 3 | 0 | 3 | **0** | 0 | 0 |
| $C_8$ | 3 | 56 | 120 | 80 / 80 | 5 | 9 | 3 | 6 | **2** | 0 | 2 |
| $C_8$ | 4 | 70 | 160 | 120 / 120 | 8 | 10 | 0 | 10 | **3** | 0 | 3 |
| $C_8$ | 5 | 56 | 120 | 80 / 80 | 5 | 9 | 3 | 6 | **2** | 0 | 2 |
| $Q_3$ | 2 | 28 | 72 | 42 / 90 | 3 | 2 | 0 | 2 | **0** | 0 | 0 |
| $Q_3$ | 3 | 56 | 180 | 168 / 306 | 3 | 6 | 3 | 3 | **1** | 0 | 1 |
| $Q_3$ | 4 | 70 | 240 | 252 / 444 | 6 | 6 | 0 | 6 | **1** | 0 | 1 |
| Petersen | 2 | 45 | 120 | 75 / 75 | 2 | 2 | 1 | 1 | **0** | 0 | 0 |
| Petersen | 3 | 120 | 420 | 450 / 450 | 4 | 6 | 2 | 4 | **1** | 0 | 1 |
| $K_9=\mathbb Z_3^2$ | 2 | 36 | 252 | 378 / 2520 | 1 | 2 | 2 | 0 | **0** | 0 | 0 |
| $K_9=\mathbb Z_3^2$ | 3 | 84 | 756 | 1890 / 10836 | 2 | 4 | 3 | 1 | **0** | 0 | 0 |
| $P_8$, $|\!\operatorname{Aut}\!|=2$ | 3 | 56 | 105 | 60 / 60 | 28 | 54 | 3 | 51 | **24** | 0 | 24 |

$\epsilon$, $\mathcal H^1_G$ and $(\operatorname{im}\delta)^G$ are **identical under both 2-cell
predicates in all 15 arenas**, so the two predicate columns collapse into one set of values.
T1, T2, T3: 30/30 ok.

---

## 6. Findings

### 6.1 The harmonic invariant channel is empty everywhere

$$\boxed{\ \mathcal H^1_G=0 \text{ in all } 30 \text{ rows — 6 base graphs, } K=2\ldots5, \text{ both predicates}\ }$$

This is the load-bearing empirical result, and it is much stronger than the three-arena version.
Consequence: in every arena measured,

$$\epsilon = \dim(\operatorname{im}\delta)^G .$$

The residual always realises as coexact. A price that charges by winding around a hole has not
appeared once.

### 6.2 Parameter 7 cannot touch the residual

$Q_3$ at $K=3$ moves $168 \to 306$ cells and $K=4$ moves $252 \to 444$; $\mathbb Z_3^2$ at $K=2$
moves $378 \to 2520$ and at $K=3$ moves $1890 \to 10836$ — a factor of 5.7. Across all of it,
$\epsilon$, $\mathcal H^1_G$ and $(\operatorname{im}\delta)^G$ do not move by one dimension.
The invariant 1-form space is a property of the 1-skeleton and the group; the 2-cells only get to
decide harmonic-versus-coexact, and on this family they never exercise that choice.

$$\boxed{\ \text{the edge residual is parameter-7-proof}\ }$$

### 6.3 The $C_8$ cases asked for, and they deliver

$$C_8:\quad \epsilon = 0,\ 2,\ 3,\ 2 \quad\text{for } K=2,3,4,5$$

$C_8$ at $K=3$ carries **two** independent invariant coexact directions, $K=4$ carries **three**.
The one-dimensional $Q_3$ result was not an accident of a one-dimensional construction.

### 6.4 Bipartiteness is dead, as expected

$C_6$ bipartite $\epsilon=0$; $C_8$ bipartite $\epsilon=2$; $Q_3$ bipartite $\epsilon=1$;
Petersen non-bipartite $\epsilon=1$; $K_9$ non-bipartite $\epsilon=0$. It separates nothing.

### 6.5 Particle–hole duality of the arena

$C_8$ at $K=3$ and $K=5$ agree in **every column**: $V$, $E$, cells, $v_G$, $e_G$, $e_G^{\rm sr}$,
$\epsilon$. Same for $C_6$ at $K=2$ and $K=4$. The arena is invariant under $K \mapsto N-K$ —
tokens and holes are the same object. Not used for anything yet; recorded.

### 6.6 Total blindness is attainable

$K_9$ at $K=2$ gives $e_G^{\rm ns}=0$: **no invariant 1-form exists at all**, not merely no
non-exact one. The arena is vertex-transitive under $\mathrm{AGL}(2,3)$ ($v_G=1$) and every edge
orbit is self-reversing. An invariant observer on that arena sees nothing on edges whatsoever.

### 6.7 A two-point correlation worth flagging, not claiming

$K=2 \Rightarrow \epsilon=0$ in all five base graphs tested. And separately, from the grading
sweep: $K=2 \Rightarrow$ the reader response can never be graded (class count $\in\{0,1\}$ for
every profile). Two different computations both die at two tokens. That may be the bridge between
the reader and arena programmes, or it may be two unrelated smallness effects. It is a
correlation on one value of one parameter and should be treated as a lead, not a result.
The discriminating test is an arena with $K=3$ and $\epsilon=0$ — $C_6$ at $K=3$ is exactly that,
and its readers **do** grade. So the two are already known not to coincide in general.
Recorded as falsified-as-identity, retained as a question about $K=2$ specifically.

---

## 7. Corrections and caveats

- **Circulation rank is not evidence.** Last run I reported 5 472 cycles with nonzero circulation
  on $Q_3$ as support for the coexact reading. Given $\epsilon=1$, nonzero circulation was forced.
  It confirms the code, nothing more. Labelled.
- **T1–T3 were presented in the crosstalk as things to test.** They are theorems under the stated
  action; the sweep is an implementation check. The experiment is §6.1 and §6.2.
- $Q_3$ and $\mathbb Z_3^2$ here are **frozen-spec** arenas, not the locked ones. Parameter 7
  remains OPEN for the $Q_3$ / $\mathbb Z_3^2$ reconciliation; §6.2 says only that the residual
  does not depend on how it is resolved.
- $\mathbb Z_3^2$ is read as Cayley$(\mathbb Z_3^2, \text{all 8 non-identity}) = K_9$ with
  $\operatorname{Aut}$ taken as $\mathrm{AGL}(2,3)$, order 432. If that reading is wrong, its two
  rows go.
- $P_8$ has $|\operatorname{Aut}|=2$, so its $\epsilon=24$ measures almost no symmetry reduction.
  It is a stress test of the identities, not a physical case.

---

## 8. Open

1. **Is there a formula for $\epsilon(X,K)$?** Current values: $C_6\!:\!0,0,0$; $C_8\!:\!0,2,3,2$;
   $Q_3\!:\!0,1,1$; Petersen$\!:\!0,1$; $K_9\!:\!0,0$. No pattern yet. This is now the cheapest
   open question in the programme — it needs no Hodge computation, only orbit counting.
2. **Can $\mathcal H^1_G \ne 0$ ever happen?** 30 rows say no. A single counterexample would
   reopen the winding channel. Candidates: base graphs with large girth and small automorphism
   group, where the arena has many holes and few of them are symmetric.
3. **The reader bridge.** Does the reader's scalar map have a kernel of dimension $\epsilon$?
   §6.7 already rules out the naive identity. The honest version of the question is whether the
   discarded sector *transforms like* the edge residual, which is a representation-theoretic
   question and not answered by dimension counting.
4. Parameter 7, unchanged: still OPEN, still not blocking anything measured here.
